In [ ]:
!pip -q install numpy
!rm -rf /content/DanyAI
!git clone -q https://github.com/DanyaVoredom/DanyAI.git /content/DanyAI
%cd /content/DanyAI
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
print("setup done")

In [ ]:
import os, json, subprocess
os.makedirs("data/raw", exist_ok=True)
ids = [1342, 11, 1661, 2701, 84, 98, 345, 5200, 16328, 2600]
for i in ids:
    subprocess.run(["curl", "-s", "-L",
                    f"https://www.gutenberg.org/cache/epub/{i}/pg{i}.txt",
                    "-o", f"data/raw/pg{i}.txt"], check=False)
print("corpus files", len(os.listdir("data/raw")))
!python -m danyai.prepare --input data/raw --out data --vocab 8192 --val 0.02
cfg = json.load(open("configs/mini.json"))
cfg.update(block_size=256, batch_size=48, steps=20000, lr=0.0015, warmup=500)
cfg["name"] = "DanyAI-Mini-Colab"
json.dump(cfg, open("configs/colab.json", "w"), indent=2)
print(cfg)
!python -m danyai.train --config configs/colab.json --data data --out runs/colab
!python -m danyai.sample --ckpt runs/colab/best.npz --tok data/tokenizer.json --prompt "The house" --tokens 200 --temp 0.8
print("train done")

import base64, pathlib
for name in ["runs/colab/best.npz", "data/tokenizer.json", "data/meta.json", "configs/colab.json"]:
    p = pathlib.Path(name)
    if not p.exists():
        print("@@MISSING", name)
        continue
    b = base64.b64encode(p.read_bytes()).decode()
    print(f"@@BEGIN {name} {len(b)}")
    for i in range(0, len(b), 50000):
        print(b[i:i + 50000])
    print(f"@@END {name}")
print("pack done")